# Topic 4 — Regression
### 3-hour seminar

**Suggested pacing (3h):**
- 0:00–0:15 — Readings/objectives
- 0:15–1:15 — Part 1: linear regression on a dummy dataset
- 1:15–1:25 — Break
- 1:25–2:45 — Part 2: regression on a real Kaggle dataset (California Housing)
- 2:45–3:00 — Wrap-up questions

This builds directly on Topic 2's EDA of the same housing dataset — revisit that notebook if needed.


## Learning objectives
- Fit and interpret a simple and multiple linear regression model
- Evaluate regression models with MSE, RMSE, MAE, and R²
- Read and diagnose a residual plot
- Build a full preprocessing + regression pipeline with scikit-learn


## Required readings
- Géron, *Hands-On Machine Learning* — Ch. 4, "Training Models" (linear regression, gradient descent) OR Raschka & Mirjalili, *Python Machine Learning* — Ch. 10, "Predicting Continuous Target Variables with Regression Analysis" (which ever you like more)
- scikit-learn linear models docs: https://scikit-learn.org/stable/modules/linear_model.html
- my Regression slides
- https://www.youtube.com/watch?v=nk2CQITm_eo

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(3)


---
## Part 1 — Dummy dataset

Synthetic data with a known ground-truth relationship, so you can check your fitted
coefficients against the truth.


In [ ]:
n = 250
x1 = np.random.uniform(0, 10, n)          # e.g. years of experience
x2 = np.random.uniform(18, 65, n)         # e.g. age
noise = np.random.normal(0, 5, n)
y = 3.0 * x1 + 0.5 * x2 + 10 + noise      # true relationship: salary-like target

df = pd.DataFrame({"years_exp": x1, "age": x2, "target": y})
df.head()


### Exercise 1.1 — Simple linear regression
1. Fit `LinearRegression` using only `years_exp` to predict `target`
2. Plot the data with the fitted line overlaid
3. Print the learned slope/intercept — how close are they to the true values (3.0, and part of 10)?


In [ ]:
# Простая регрессия: используем только years_exp
X_simple = df[["years_exp"]]
y_target = df["target"]
simple_model = LinearRegression()
simple_model.fit(X_simple, y_target)

x_line = np.linspace(df["years_exp"].min(), df["years_exp"].max(), 100)
y_line = simple_model.predict(pd.DataFrame({"years_exp": x_line}))
plt.figure(figsize=(8, 5))
plt.scatter(df["years_exp"], df["target"], alpha=0.6, label="Observations")
plt.plot(x_line, y_line, color="red", linewidth=2, label="Fitted line")
plt.xlabel("Years of experience")
plt.ylabel("Target")
plt.title("Simple linear regression")
plt.legend()
plt.show()

print(f"Slope: {simple_model.coef_[0]:.3f}")
print(f"Intercept: {simple_model.intercept_:.3f}")
print("В простой модели коэффициент при опыте не обязан быть ровно 3: возраст тоже влияет на target и не включён в эту модель.")

### Exercise 1.2 — Multiple linear regression
1. Fit `LinearRegression` using both `years_exp` and `age`
2. Split into train/test (80/20)
3. Report RMSE, MAE, and R² on the test set
4. Compare the learned coefficients to the true values (3.0 and 0.5)


In [ ]:
# Разделяем признаки и цель; тестовые данные не участвуют в обучении
X = df[["years_exp", "age"]]
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
multiple_model = LinearRegression()
multiple_model.fit(X_train, y_train)
y_pred = multiple_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Коэффициент years_exp: {multiple_model.coef_[0]:.3f} (истинный 3.0)")
print(f"Коэффициент age: {multiple_model.coef_[1]:.3f} (истинный 0.5)")
print(f"Intercept: {multiple_model.intercept_:.3f} (истинный 10.0)")
print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE: {mae:.3f}")
print(f"Test R²: {r2:.3f}")

### Exercise 1.3 — Residual plot
Plot predicted values vs. residuals (`y_true - y_pred`) for your test set.
A good fit should show residuals scattered randomly around zero with no pattern —
describe what you see.


In [ ]:
residuals = y_test - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted values")
plt.ylabel("Residuals (actual - predicted)")
plt.title("Residual plot")
plt.show()
print("Остатки должны быть разбросаны вокруг нуля без выраженного рисунка; на синтетических данных ожидается примерно случайный разброс.")

Остатки расположены вокруг нулевой линии без очевидной кривизны или расширяющегося веера. Это согласуется с линейной моделью и примерно постоянной дисперсией ошибок. Случайный график не доказывает, что все предположения модели выполнены, но явного нарушения здесь не видно.

---
## Part 2 — Real data: California Housing Prices (Kaggle)

**Get the data:** https://www.kaggle.com/datasets/camnugent/california-housing-prices
(same file as Topic 2 — `housing.csv`)


In [ ]:
from pathlib import Path
DATA_PATH = "housing.csv"  # положите Kaggle housing.csv рядом с ноутбуком
if not Path(DATA_PATH).exists():
    raise FileNotFoundError(
        "Не найден housing.csv. Скачайте California Housing Prices housing.csv "
        "из Kaggle и положите его в ту же папку, что и ноутбук."
    )
housing = pd.read_csv(DATA_PATH)
housing = housing.dropna(subset=["total_bedrooms"]).copy()
housing.head()

### Exercise 2.1 — Build a preprocessing + regression pipeline
1. Separate features `X` (drop `median_house_value`) and target `y`
2. Split numeric vs. categorical columns (`ocean_proximity` is categorical)
3. Build a `ColumnTransformer` that scales numeric features (`StandardScaler`) and
   one-hot encodes the categorical feature
4. Wrap it in a `Pipeline` with a `LinearRegression` at the end


In [ ]:
# Целевая переменная — стоимость жилья; остальные столбцы используются как признаки
X = housing.drop(columns=["median_house_value"])
y = housing["median_house_value"]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])
print("Числовые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)

### Exercise 2.2 — Train, predict, evaluate
1. Train/test split (80/20, `random_state=42`)
2. Fit the pipeline on the training data
3. Report RMSE, MAE, R² on the test set
4. Compare RMSE against a trivial baseline that always predicts the training mean


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Бейзлайн: всем тестовым объектам предсказываем среднее целевой переменной в train
baseline_pred = np.full(shape=len(y_test), fill_value=y_train.mean(), dtype=float)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
print(f"Pipeline RMSE: {rmse:,.2f}")
print(f"Pipeline MAE: {mae:,.2f}")
print(f"Pipeline R²: {r2:.3f}")
print(f"Baseline RMSE (training mean): {baseline_rmse:,.2f}")
print("Модель лучше среднего бейзлайна по RMSE." if rmse < baseline_rmse else "Модель не превзошла средний бейзлайн по RMSE.")

### Exercise 2.3 — Which features matter?
Extract the fitted `LinearRegression` coefficients from the pipeline (careful — they line
up with the *transformed* feature names, not the original ones). Which features have the
largest absolute coefficients?


In [ ]:
fitted_preprocessor = model_pipeline.named_steps["preprocessor"]
fitted_regressor = model_pipeline.named_steps["regressor"]
feature_names = fitted_preprocessor.get_feature_names_out()
coef_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": fitted_regressor.coef_,
})
coef_table["abs_coefficient"] = coef_table["coefficient"].abs()
print("Признаки с наибольшими коэффициентами по абсолютной величине:")
print(coef_table.sort_values("abs_coefficient", ascending=False).head(10).to_string(index=False))
print("\nЧисловые признаки были стандартизированы, поэтому их коэффициенты относятся к изменению признака на одно стандартное отклонение. Категориальные коэффициенты соответствуют one-hot индикаторам относительно базовой категории.")

## Ответы на вопросы

1. Масштабирование приводит числовые признаки к сопоставимому масштабу. Для обычной линейной регрессии с решением методом наименьших квадратов оно не обязательно для самого качества предсказания, но облегчает сравнение коэффициентов и часто важно для численно чувствительных/регуляризованных моделей.
2. Если оценивать модель на тех же данных, на которых она обучалась, R² обычно будет завышен относительно качества на новых данных: модель уже подстроилась под эту выборку.
3. Пример нелинейности: влияние расстояния до океана или возраста жилья может быть нелинейным; также возможны взаимодействия между доходом, размером домохозяйства и географическим положением.